<a href="https://colab.research.google.com/github/rafidfajar/data-science-2026/blob/main/Pertemuan12_Muhamad_Rafid_Fajar_250401020195.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Nama    :** Muhamad Rafid Fajar
### **NIM     :** 250401020195
### **Kelas   :** IF401

In [18]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']
# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
 n_item = np.random.randint(2, 6)
 transaksi.append(list(np.random.choice(produk, n_item, replace=False)))
# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
 if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
    transaksi[i].append('Selai')
print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))


Contoh transaksi: [[np.str_('Roti'), np.str_('Sereal'), np.str_('Susu'), 'Selai'], [np.str_('Teh'), np.str_('Gula'), np.str_('Selai'), np.str_('Sereal')], [np.str_('Roti'), np.str_('Gula'), 'Selai']]
Jumlah transaksi: 50


In [2]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

In [19]:
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
 freq = apriori(df, min_support=ms, use_colnames=True)
 print(f'min_support={ms}: {len(freq)} itemset ditemukan')
# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))


min_support=0.05: 101 itemset ditemukan
min_support=0.1: 56 itemset ditemukan
min_support=0.2: 15 itemset ditemukan
   support   itemsets
5     0.52    (Selai)
1     0.48     (Keju)
7     0.42     (Susu)
2     0.38     (Kopi)
9     0.36    (Telur)
0     0.34     (Gula)
4     0.30     (Roti)
3     0.30  (Mentega)
6     0.30   (Sereal)
8     0.28      (Teh)


In [20]:
from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
 min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
 'support', 'confidence', 'lift']].head(10))
# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?


        antecedents consequents  support  confidence      lift
21    (Gula, Selai)       (Teh)     0.10    0.625000  2.232143
16  (Selai, Sereal)      (Roti)     0.12    0.666667  2.222222
19    (Gula, Selai)      (Roti)     0.10    0.625000  2.083333
18   (Sereal, Roti)     (Selai)     0.12    1.000000  1.923077
20     (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
10    (Selai, Susu)      (Roti)     0.12    0.545455  1.818182
26     (Keju, Kopi)      (Susu)     0.10    0.714286  1.700680
9             (Teh)      (Gula)     0.16    0.571429  1.680672
28    (Selai, Kopi)    (Sereal)     0.10    0.500000  1.666667
17    (Selai, Roti)    (Sereal)     0.12    0.500000  1.666667


Aturan mana yang paling kuat (Lift tertinggi)?
(Selai, Susu) dengan consequents(Mentega), dengan lift 1,52 dan confidence 0,64.

Apakah masuk akal secara bisnis?
Ya, cukup masuk akal Selai, Susu, dan Mentega sama-sama produk sarapan/dairy yang wajar dibeli bersamaan

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
 'produk': produk,
 'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)
def rekomendasi_serupa(nama_produk, top_n=3):
 idx = katalog.index[katalog['produk'] == nama_produk][0]
 skor = list(enumerate(sim_matrix[idx]))
 skor = sorted(skor, key=lambda x: x[1], reverse=True)
 skor = [s for s in skor if s[0] != idx][:top_n]
 return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))


Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [22]:
produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
 lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))
# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?


Rekomendasi dari Association Rules:
   consequents      lift
18     (Selai)  1.923077
20     (Selai)  1.923077
17    (Sereal)  1.666667
12     (Selai)  1.648352
24     (Selai)  1.602564
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


**Apakah kedua pendekatan konsisten?**
Tidak sepenuhnya. Association rules merekomendasikan produk berdasarkan **pola pembelian aktual** (mis. Roti → Selai berdasarkan transaksi nyata), sedangkan content-based merekomendasikan berdasarkan **kemiripan kategori** (Roti → Selai, Sereal, Susu, karena sama-sama Bakery/Dairy). Selai kebetulan muncul di kedua metode, tapi Sereal dan Susu hanya muncul di content-based — jadi hasilnya bisa berbeda arah tergantung metode yang dipakai.

**Kapan pakai yang mana / hybrid?**
- **Association rules**: cocok kalau data transaksi banyak dan pola beli-bersama sudah teruji secara statistik.
- **Content-based**: cocok untuk produk baru atau *cold-start* yang belum punya riwayat transaksi.
- **Hybrid**: gabungkan keduanya — utamakan association rules jika datanya kuat, fallback ke content-based jika data transaksi tipis atau produk baru. Ini pendekatan umum di sistem rekomendasi nyata.